# LoRA fine-tune ESM-2 650M on UniProtSMB (Colab **free** T4)

The flagship modern run. A free Colab **T4 (16 GB)** fits `esm2_t33_650M_UR50D` with LoRA +
gradient checkpointing, which does **not** fit the local 4 GB GTX 1650.

**Before running:** Runtime -> Change runtime type -> **GPU (T4)**, then Runtime -> Run all.

**Free-tier notes (handled below):**
- Free sessions disconnect after ~90 min idle and cap around ~12 h. This run is a few hours of
  *active* training, so keep the tab focused.
- We mount Google Drive and write checkpoints there on every val improvement, so a disconnect
  never loses the best adapter — just re-run and it resumes from the saved best.
- All metrics logged here are measured on the held-out test split; nothing is hand-entered.

In [ ]:
import torch; assert torch.cuda.is_available(), 'Enable GPU: Runtime > Change runtime type > T4'
print(torch.cuda.get_device_name(0))  # expect Tesla T4

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/pbsite_outputs'  # checkpoints persist here
import os; os.makedirs(OUT, exist_ok=True)

In [ ]:
# Public repo — no token needed. Lean install: Colab already ships torch/transformers/etc,
# so we only add what's missing instead of re-resolving all of requirements.txt.
%cd /content
!rm -rf protein-binding-esm-lora
!git clone https://github.com/aakashshahani/protein-binding-esm-lora.git
%cd protein-binding-esm-lora
!test -f requirements.txt && echo 'CLONE OK' || echo 'CLONE FAILED'
!pip -q install peft accelerate wandb
# Colab ships an old torchao (0.10) that breaks peft's LoRA dispatch; we don't use it.
!pip -q uninstall -y torchao
!pip -q install -e . --no-deps

In [ ]:
# Fetch the exact CLAPE-SMB / UniProtSMB split (pinned commit) + data card.
!python scripts/download_data.py --config configs/data.yaml
!python scripts/build_dataset.py --config configs/data.yaml

In [ ]:
import os
os.environ['WANDB_MODE'] = 'offline'  # set 'online' + WANDB_API_KEY to sync
# 650M LoRA, colab track (batch 4, grad accum 4, grad checkpointing, max_len 1022).
# --out-dir on Drive so the best adapter survives a disconnect.
!python scripts/train.py --config configs/lora.yaml --track colab --out-dir "$OUT"

In [ ]:
# Evaluate the 650M LoRA on the held-out test set -> real numbers for the README.
RUN = f"{OUT}/lora_esm2_t33_650M_UR50D_colab"
!python scripts/evaluate_lora.py --run "$RUN" --model-id facebook/esm2_t33_650M_UR50D --track colab --win 1020
import json; print(json.load(open(f'{RUN}/test_metrics.json')))

Copy the printed test metrics into the README results table (row: *LoRA ESM-2 650M (Colab)*).
The adapter + head under `RUN` on your Drive can be downloaded and served with the FastAPI app.